# Detector de mascarillas: usando el modelo que ajustamos en Roboflow

El sábado se hizo fine-tuning de un **RF-DETR Small** en Roboflow con dos clases: `Mask` y `NO-Mask`. Este notebook muestra cómo **usar** ese modelo desde Python y cómo armar una app con Gradio para probarlo con fotos o la cámara.

## 🔐 Antes de empezar: la clave de Roboflow va en Secretos, nunca en el código

1. En la barra izquierda de Colab, haz clic en el ícono de la **llave** (🔑 Secretos).
2. **Agregar nuevo secreto** con el nombre `ROBOFLOW_API_KEY` y tu clave como valor.
3. Activa el interruptor **Acceso al notebook**.

Los secretos se guardan en tu cuenta de Google, no en el notebook. Por eso este archivo se puede subir a GitHub sin exponer la clave.

Tu clave está en Roboflow → Settings → API Keys (usa la *Private API Key*).

In [ ]:
!pip install -q inference-sdk gradio

## 1. Conectarse al modelo sin escribir la clave

In [ ]:
from inference_sdk import InferenceHTTPClient, InferenceConfiguration

def obtener_clave():
    try:
        from google.colab import userdata
        return userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        # Si no hay secreto configurado, se pide por teclado y no queda guardada en el notebook
        from getpass import getpass
        return getpass("Pega tu ROBOFLOW_API_KEY (no se mostrará): ")

API_KEY = obtener_clave()

# Identificador del modelo entrenado (no es secreto)
MODEL_ID = "innovacien/mask-wearing-iskms-nmeob-1-rfdetr-small-t1"
# Si da error de modelo no encontrado, prueba con: "mask-wearing-iskms-nmeob/1"

client = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key=API_KEY,
).configure(InferenceConfiguration(api_key_transport="header"))

print("✅ Cliente listo. La clave tiene", len(API_KEY), "caracteres (nunca la imprimas completa).")

## 2. Primera predicción: ¿qué devuelve el modelo?

Sube una foto con personas. El modelo devuelve una lista de **predicciones**, cada una con la posición del recuadro (`x`, `y`, `width`, `height`), la clase y la confianza.

In [ ]:
from google.colab import files
from PIL import Image
import io

subida = files.upload()
nombre = next(iter(subida))
foto = Image.open(io.BytesIO(subida[nombre])).convert("RGB")

resultado = client.infer(foto, model_id=MODEL_ID)

print(f"Se detectaron {len(resultado['predictions'])} rostros\n")
for p in resultado["predictions"]:
    print(f"  {p['class']:8s}  confianza {p['confidence']:.0%}   centro=({p['x']:.0f}, {p['y']:.0f})  tamaño={p['width']:.0f}×{p['height']:.0f}")

## 3. Dibujar los recuadros

In [ ]:
from PIL import ImageDraw, ImageFont

try:
    FUENTE = ImageFont.load_default(size=18)
except TypeError:
    FUENTE = ImageFont.load_default()

COLORES = {"Mask": "#18a558", "NO-Mask": "#e63946"}
NOMBRES = {"Mask": "Con mascarilla", "NO-Mask": "Sin mascarilla"}

def dibujar(imagen, predicciones, confianza_minima=0.5):
    salida = imagen.copy()
    draw = ImageDraw.Draw(salida)
    conteo = {"Mask": 0, "NO-Mask": 0}

    for p in predicciones:
        clase, conf = p.get("class"), float(p.get("confidence", 0))
        if clase not in conteo or conf < confianza_minima:
            continue
        conteo[clase] += 1

        # Roboflow entrega el CENTRO del recuadro; se convierte a esquinas
        x, y, w, h = p["x"], p["y"], p["width"], p["height"]
        caja = (max(0, x - w / 2), max(0, y - h / 2),
                min(salida.width - 1, x + w / 2), min(salida.height - 1, y + h / 2))
        color = COLORES[clase]
        texto = f"{NOMBRES[clase]} {conf:.0%}"

        draw.rectangle(caja, outline=color, width=4)
        t = draw.textbbox((0, 0), texto, font=FUENTE)
        alto = t[3] - t[1]
        arriba = max(0, caja[1] - alto - 8)
        draw.rectangle((caja[0], arriba, caja[0] + t[2] - t[0] + 8, arriba + alto + 8), fill=color)
        draw.text((caja[0] + 4, arriba + 2), texto, fill="white", font=FUENTE)

    return salida, conteo

imagen_marcada, conteo = dibujar(foto, resultado["predictions"])
print(conteo)
imagen_marcada

## 4. App con Gradio

`share=True` genera un enlace público temporal: los estudiantes pueden abrirlo en el celular y probar con su cámara.

⚠️ La app corre en este Colab y usa **tu** clave en el servidor (el enlace no la muestra), pero cada foto que suban gasta créditos de tu cuenta. Detén la celda al terminar la clase y el enlace deja de funcionar.

In [ ]:
import gradio as gr

def analizar(imagen, confianza_minima):
    if imagen is None:
        return None, "Sube una foto o toma una con la cámara."
    imagen = imagen.convert("RGB")
    resultado = client.infer(imagen, model_id=MODEL_ID)
    salida, conteo = dibujar(imagen, resultado.get("predictions", []), confianza_minima)
    if sum(conteo.values()) == 0:
        return salida, "No se detectaron rostros con suficiente confianza: resultado indeterminado."
    return salida, f"😷 Con mascarilla: {conteo['Mask']}   ·   🙂 Sin mascarilla: {conteo['NO-Mask']}"

demo = gr.Interface(
    fn=analizar,
    inputs=[gr.Image(type="pil", sources=["upload", "webcam"], label="Foto"),
            gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Confianza mínima")],
    outputs=[gr.Image(type="pil", label="Detecciones"), gr.Textbox(label="Resultado")],
    title="Detección de uso de mascarilla",
    description="Modelo RF-DETR Small ajustado en Roboflow por el curso. Sube una foto o usa la cámara.",
)
demo.launch(share=True, debug=True)

## 5. Antes de subir a GitHub

1. **Editar → Borrar todos los resultados** (las salidas pueden incluir fotos de personas y el enlace público).
2. **Archivo → Guardar una copia en GitHub**.
3. Revisa en GitHub que no aparezca la clave en ninguna parte. Si alguna vez se filtra, genera una nueva en Roboflow y revoca la anterior.

**Para discutir en clase:** mueve el control de confianza mínima y observa qué pasa con fotos difíciles (perfil, poca luz, bufandas). ¿Qué errores comete el modelo? ¿Qué fotos faltarían en el dataset para mejorarlo?